# TimeMMD Colab Benchmark

## Goal

Run the Aurora TimeMMD benchmark for Chronos-2 and Chronos-2-ECHO in Colab without a local repository checkout. The notebook installs the public `chronos2-echo` package from GitHub, defines the benchmark manifest and helper code inline, validates the Aurora-compatible CSV files, writes result artifacts, and packages the run directory as a zip.

Use the Aurora benchmark data, not a locally re-merged Time-MMD export. Aurora's official repo links the benchmark bundle here:
https://drive.google.com/file/d/12tJk858WaoG7ZVSvUq8KU1oHfGNJrARF/view?usp=drive_link

Expected CSV files under `DATA_ROOT`: `Agriculture.csv`, `Climate.csv`, `Economy.csv`, `Energy.csv`, `Environment.csv`, `Health.csv`, `Security.csv`, `Traffic.csv`, and `SocialGood.csv`.

## Setup

In [ ]:
import subprocess
import sys

PACKAGE_SPEC = "chronos2-echo[timemmd] @ git+https://github.com/2018wzh/Chronos-ECHO.git"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", PACKAGE_SPEC])
print("Installed", PACKAGE_SPEC)

In [ ]:
from pathlib import Path
import torch

print("python", sys.version.split()[0])
print("torch", torch.__version__)
print("cuda_available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

## Parameters

In [ ]:
from pathlib import Path
import time

DATA_ROOT = Path("TimeMMD/dataset")  # @param {type:"string"}
RUN_ID = time.strftime("%Y%m%d-%H%M%S")
OUTPUT_DIR = Path("timemmd_runs") / RUN_ID  # @param {type:"string"}
MODELS = "chronos2,echo_zero_shot"  # @param ["chronos2", "echo_zero_shot", "chronos2,echo_zero_shot", "echo_few_shot"] {allow-input: true}
CHRONOS_MODEL = "amazon/chronos-2"  # @param {type:"string"}
ECHO_BASE_MODEL = "amazon/chronos-2"  # @param {type:"string"}
TEXT_TOKENIZER = "bert-base-uncased"  # @param {type:"string"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 2021  # @param {type:"integer"}
BATCH_SIZE = None
SAVE_TO_DRIVE = False  # @param {type:"boolean"}
DRIVE_OUTPUT_ROOT = Path("drive/MyDrive/timemmd-benchmark")

FEWSHOT_LR = 5e-6
FEWSHOT_STEPS = 1000
FEWSHOT_WARMUP_STEPS = 100
FEWSHOT_BATCH_SIZE = 8
FEWSHOT_LR_SCHEDULER = "cosine"
FORCE_RETRAIN = False
CHECKPOINT_ROOT = Path("timemmd_checkpoints/chronos2_echo_fewshot")
ECHO_CONFIG_OVERRIDES = {}

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("drive")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR = DRIVE_OUTPUT_ROOT / RUN_ID
    CHECKPOINT_ROOT = DRIVE_OUTPUT_ROOT / "checkpoints" / "chronos2_echo_fewshot"

print("DATA_ROOT", DATA_ROOT)
print("OUTPUT_DIR", OUTPUT_DIR)
print("MODELS", MODELS)
print("DEVICE", DEVICE)

## Benchmark Helpers

In [ ]:
MANIFEST_CSV = 'domain,data_path,seq_len,pred_len,features,target,text_column,split,inference_token_len,batch_size\nAgriculture,Agriculture.csv,192,6,S,OT,fact,test,48,256\nAgriculture,Agriculture.csv,192,8,S,OT,fact,test,48,256\nAgriculture,Agriculture.csv,192,10,S,OT,fact,test,48,256\nAgriculture,Agriculture.csv,192,12,S,OT,fact,test,48,256\nClimate,Climate.csv,192,6,S,OT,fact,test,48,256\nClimate,Climate.csv,192,8,S,OT,fact,test,48,256\nClimate,Climate.csv,192,10,S,OT,fact,test,48,256\nClimate,Climate.csv,192,12,S,OT,fact,test,48,256\nEconomy,Economy.csv,192,6,S,OT,fact,test,48,256\nEconomy,Economy.csv,192,8,S,OT,fact,test,48,256\nEconomy,Economy.csv,192,10,S,OT,fact,test,48,256\nEconomy,Economy.csv,192,12,S,OT,fact,test,48,256\nEnergy,Energy.csv,1056,12,S,OT,fact,test,48,256\nEnergy,Energy.csv,1056,24,S,OT,fact,test,48,256\nEnergy,Energy.csv,1056,36,S,OT,fact,test,48,256\nEnergy,Energy.csv,1056,48,S,OT,fact,test,48,256\nEnvironment,Environment.csv,528,48,S,OT,fact,test,48,256\nEnvironment,Environment.csv,528,96,S,OT,fact,test,48,256\nEnvironment,Environment.csv,528,192,S,OT,fact,test,48,256\nEnvironment,Environment.csv,528,336,S,OT,fact,test,48,256\nHealth,Health.csv,96,12,S,OT,fact,test,48,256\nHealth,Health.csv,96,24,S,OT,fact,test,48,256\nHealth,Health.csv,96,36,S,OT,fact,test,48,256\nHealth,Health.csv,96,48,S,OT,fact,test,48,256\nSecurity,Security.csv,220,6,S,OT,fact,test,24,256\nSecurity,Security.csv,220,8,S,OT,fact,test,24,256\nSecurity,Security.csv,220,10,S,OT,fact,test,24,256\nSecurity,Security.csv,220,12,S,OT,fact,test,24,256\nTraffic,Traffic.csv,96,6,S,OT,fact,test,48,256\nTraffic,Traffic.csv,96,8,S,OT,fact,test,48,256\nTraffic,Traffic.csv,96,10,S,OT,fact,test,48,256\nTraffic,Traffic.csv,96,12,S,OT,fact,test,48,256\nSocialGood,SocialGood.csv,192,6,S,OT,fact,test,48,256\nSocialGood,SocialGood.csv,192,8,S,OT,fact,test,48,256\nSocialGood,SocialGood.csv,192,10,S,OT,fact,test,48,256\nSocialGood,SocialGood.csv,192,12,S,OT,fact,test,48,256\n'

AURORA_SOURCE_URL = 'https://arxiv.org/html/2509.22295v4'
AURORA_REFERENCE = {('Agriculture', 6, 'aurora_few_shot'): {'mae': 0.233, 'mse': 0.127},
 ('Agriculture', 6, 'aurora_zero_shot'): {'mae': 0.295, 'mse': 0.184},
 ('Agriculture', 8, 'aurora_few_shot'): {'mae': 0.289, 'mse': 0.19},
 ('Agriculture', 8, 'aurora_zero_shot'): {'mae': 0.335, 'mse': 0.242},
 ('Agriculture', 10, 'aurora_few_shot'): {'mae': 0.31, 'mse': 0.236},
 ('Agriculture', 10, 'aurora_zero_shot'): {'mae': 0.365, 'mse': 0.297},
 ('Agriculture', 12, 'aurora_few_shot'): {'mae': 0.34, 'mse': 0.295},
 ('Agriculture', 12, 'aurora_zero_shot'): {'mae': 0.398, 'mse': 0.365},
 ('Climate', 6, 'aurora_few_shot'): {'mae': 0.744, 'mse': 0.867},
 ('Climate', 6, 'aurora_zero_shot'): {'mae': 0.747, 'mse': 0.859},
 ('Climate', 8, 'aurora_few_shot'): {'mae': 0.745, 'mse': 0.858},
 ('Climate', 8, 'aurora_zero_shot'): {'mae': 0.746, 'mse': 0.858},
 ('Climate', 10, 'aurora_few_shot'): {'mae': 0.744, 'mse': 0.863},
 ('Climate', 10, 'aurora_zero_shot'): {'mae': 0.748, 'mse': 0.868},
 ('Climate', 12, 'aurora_few_shot'): {'mae': 0.749, 'mse': 0.869},
 ('Climate', 12, 'aurora_zero_shot'): {'mae': 0.753, 'mse': 0.875},
 ('Economy', 6, 'aurora_few_shot'): {'mae': 0.095, 'mse': 0.015},
 ('Economy', 6, 'aurora_zero_shot'): {'mae': 0.15, 'mse': 0.035},
 ('Economy', 8, 'aurora_few_shot'): {'mae': 0.099, 'mse': 0.015},
 ('Economy', 8, 'aurora_zero_shot'): {'mae': 0.145, 'mse': 0.033},
 ('Economy', 10, 'aurora_few_shot'): {'mae': 0.101, 'mse': 0.016},
 ('Economy', 10, 'aurora_zero_shot'): {'mae': 0.143, 'mse': 0.032},
 ('Economy', 12, 'aurora_few_shot'): {'mae': 0.102, 'mse': 0.016},
 ('Economy', 12, 'aurora_zero_shot'): {'mae': 0.144, 'mse': 0.032},
 ('Energy', 12, 'aurora_few_shot'): {'mae': 0.212, 'mse': 0.097},
 ('Energy', 12, 'aurora_zero_shot'): {'mae': 0.245, 'mse': 0.117},
 ('Energy', 24, 'aurora_few_shot'): {'mae': 0.322, 'mse': 0.199},
 ('Energy', 24, 'aurora_zero_shot'): {'mae': 0.354, 'mse': 0.226},
 ('Energy', 36, 'aurora_few_shot'): {'mae': 0.352, 'mse': 0.271},
 ('Energy', 36, 'aurora_zero_shot'): {'mae': 0.409, 'mse': 0.292},
 ('Energy', 48, 'aurora_few_shot'): {'mae': 0.431, 'mse': 0.352},
 ('Energy', 48, 'aurora_zero_shot'): {'mae': 0.472, 'mse': 0.383},
 ('Environment', 48, 'aurora_few_shot'): {'mae': 0.372, 'mse': 0.269},
 ('Environment', 48, 'aurora_zero_shot'): {'mae': 0.38, 'mse': 0.281},
 ('Environment', 96, 'aurora_few_shot'): {'mae': 0.373, 'mse': 0.271},
 ('Environment', 96, 'aurora_zero_shot'): {'mae': 0.382, 'mse': 0.284},
 ('Environment', 192, 'aurora_few_shot'): {'mae': 0.374, 'mse': 0.269},
 ('Environment', 192, 'aurora_zero_shot'): {'mae': 0.375, 'mse': 0.27},
 ('Environment', 336, 'aurora_few_shot'): {'mae': 0.368, 'mse': 0.251},
 ('Environment', 336, 'aurora_zero_shot'): {'mae': 0.377, 'mse': 0.269},
 ('Health', 12, 'aurora_few_shot'): {'mae': 0.641, 'mse': 0.992},
 ('Health', 12, 'aurora_zero_shot'): {'mae': 0.668, 'mse': 1.093},
 ('Health', 24, 'aurora_few_shot'): {'mae': 0.796, 'mse': 1.332},
 ('Health', 24, 'aurora_zero_shot'): {'mae': 0.849, 'mse': 1.572},
 ('Health', 36, 'aurora_few_shot'): {'mae': 0.818, 'mse': 1.467},
 ('Health', 36, 'aurora_zero_shot'): {'mae': 0.92, 'mse': 1.688},
 ('Health', 48, 'aurora_few_shot'): {'mae': 0.847, 'mse': 1.579},
 ('Health', 48, 'aurora_zero_shot'): {'mae': 0.963, 'mse': 1.857},
 ('Security', 6, 'aurora_few_shot'): {'mae': 3.798, 'mse': 64.513},
 ('Security', 6, 'aurora_zero_shot'): {'mae': 3.909, 'mse': 67.572},
 ('Security', 8, 'aurora_few_shot'): {'mae': 3.93, 'mse': 67.828},
 ('Security', 8, 'aurora_zero_shot'): {'mae': 4.013, 'mse': 70.576},
 ('Security', 10, 'aurora_few_shot'): {'mae': 4.092, 'mse': 72.423},
 ('Security', 10, 'aurora_zero_shot'): {'mae': 4.148, 'mse': 74.173},
 ('Security', 12, 'aurora_few_shot'): {'mae': 4.132, 'mse': 75.482},
 ('Security', 12, 'aurora_zero_shot'): {'mae': 4.264, 'mse': 77.579},
 ('SocialGood', 6, 'aurora_few_shot'): {'mae': 0.427, 'mse': 0.689},
 ('SocialGood', 6, 'aurora_zero_shot'): {'mae': 0.442, 'mse': 0.701},
 ('SocialGood', 8, 'aurora_few_shot'): {'mae': 0.461, 'mse': 0.784},
 ('SocialGood', 8, 'aurora_zero_shot'): {'mae': 0.493, 'mse': 0.804},
 ('SocialGood', 10, 'aurora_few_shot'): {'mae': 0.532, 'mse': 0.85},
 ('SocialGood', 10, 'aurora_zero_shot'): {'mae': 0.543, 'mse': 0.886},
 ('SocialGood', 12, 'aurora_few_shot'): {'mae': 0.554, 'mse': 0.931},
 ('SocialGood', 12, 'aurora_zero_shot'): {'mae': 0.587, 'mse': 0.96},
 ('Traffic', 6, 'aurora_few_shot'): {'mae': 0.292, 'mse': 0.149},
 ('Traffic', 6, 'aurora_zero_shot'): {'mae': 0.285, 'mse': 0.154},
 ('Traffic', 8, 'aurora_few_shot'): {'mae': 0.284, 'mse': 0.155},
 ('Traffic', 8, 'aurora_zero_shot'): {'mae': 0.286, 'mse': 0.158},
 ('Traffic', 10, 'aurora_few_shot'): {'mae': 0.287, 'mse': 0.16},
 ('Traffic', 10, 'aurora_zero_shot'): {'mae': 0.289, 'mse': 0.163},
 ('Traffic', 12, 'aurora_few_shot'): {'mae': 0.296, 'mse': 0.165},
 ('Traffic', 12, 'aurora_zero_shot'): {'mae': 0.294, 'mse': 0.168}}

def get_reference(domain: str, pred_len: int, reference_model: str) -> dict[str, float] | None:
    return AURORA_REFERENCE.get((domain, int(pred_len), reference_model))

In [ ]:
from __future__ import annotations

import math
from pathlib import Path
from typing import Any, Iterator

import numpy as np
import torch
from torch.utils.data import Dataset, IterableDataset

AURORA_MISSING_TEXT = "No information available"


def create_timemmd_tokenizer(tokenizer_name_or_path: str | None) -> Any:
    if tokenizer_name_or_path is None:
        raise ValueError("Set text_tokenizer_name_or_path or pass an explicit tokenizer for TimeMMD text input.")

    from transformers import AutoTokenizer

    return AutoTokenizer.from_pretrained(tokenizer_name_or_path)


class TimeMMDWindowDataset(Dataset):
    required_columns = {"date", "prior_history_avg", "start_date", "end_date"}

    def __init__(
        self,
        *,
        root_path: str | Path,
        data_path: str,
        flag: str = "train",
        seq_len: int,
        pred_len: int,
        target: str = "OT",
        features: str = "S",
        tokenizer: Any | None = None,
        max_text_length: int = 500,
        text_column: str = "fact",
        image_column: str | None = None,
        image_root_path: str | Path | None = None,
        image_size: int = 64,
        scale: bool = True,
        few_shot_ratio: float = 0.1,
    ) -> None:
        super().__init__()
        if flag not in {"train", "val", "test", "fewshot"}:
            raise ValueError("flag must be one of 'train', 'val', 'test', or 'fewshot'")
        if features not in {"S", "M", "MS"}:
            raise ValueError("features must be one of 'S', 'M', or 'MS'")
        if tokenizer is None:
            raise ValueError("TimeMMDWindowDataset requires an explicit tokenizer")

        self.root_path = Path(root_path)
        self.data_path = data_path
        self.flag = flag
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.target = target
        self.features = features
        self.tokenizer = tokenizer
        self.max_text_length = max_text_length
        self.text_column = text_column
        self.image_column = image_column
        if image_root_path is None:
            self.image_root_path = self.root_path
        else:
            image_root_path = Path(image_root_path)
            self.image_root_path = image_root_path if image_root_path.is_absolute() else self.root_path / image_root_path
        self.image_size = image_size
        self.scale = scale
        self.few_shot_ratio = few_shot_ratio
        self._read_data()

    def _read_data(self) -> None:
        import pandas as pd

        df = pd.read_csv(self.root_path / self.data_path)
        missing = self.required_columns - set(df.columns)
        if missing:
            raise ValueError(f"TimeMMD CSV is missing required columns: {sorted(missing)}")
        if self.target not in df.columns:
            raise ValueError(f"Target column {self.target!r} not found in TimeMMD CSV")
        if self.text_column not in df.columns:
            raise ValueError(f"Text column {self.text_column!r} not found in TimeMMD CSV")
        if self.image_column is not None and self.image_column not in df.columns:
            raise ValueError(f"Image column {self.image_column!r} not found in TimeMMD CSV")

        metadata_columns = self.required_columns | {"date", self.text_column}
        if self.image_column is not None:
            metadata_columns.add(self.image_column)
        numeric_columns = []
        for col in [col for col in df.columns if col not in metadata_columns]:
            series = pd.to_numeric(df[col], errors="coerce")
            if series.notna().any():
                df[col] = series
                numeric_columns.append(col)
        if self.target not in numeric_columns:
            raise ValueError(f"Target column {self.target!r} must be numeric")

        if self.features == "S":
            self.target_columns = [self.target]
            self.covariate_columns: list[str] = []
        elif self.features == "M":
            self.target_columns = numeric_columns
            self.covariate_columns = []
        else:
            self.target_columns = [self.target]
            self.covariate_columns = [col for col in numeric_columns if col != self.target and col != "prior_history_avg"]

        value_columns = self.target_columns + self.covariate_columns
        values = df[value_columns].to_numpy(dtype=np.float32)
        prior = pd.to_numeric(df["prior_history_avg"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)
        text_series = df[self.text_column].fillna(AURORA_MISSING_TEXT).astype(str)
        if text_series.str.strip().eq("").any():
            raise ValueError(
                f"TimeMMD CSV contains empty {self.text_column} values; text must be provided explicitly"
            )
        text = text_series.to_numpy()
        image_paths = df[self.image_column].fillna("").astype(str).to_numpy() if self.image_column else None

        num_train = int(len(df) * 0.7)
        num_test = int(len(df) * 0.2)
        num_val = len(df) - num_train - num_test
        border1s = [0, num_train - self.seq_len, len(df) - num_test - self.seq_len]
        border2s = [num_train, num_train + num_val, len(df)]

        if self.flag == "train":
            border1, border2 = border1s[0], border2s[0]
        elif self.flag == "val":
            border1, border2 = border1s[1], border2s[1]
        elif self.flag == "test":
            border1, border2 = border1s[2], border2s[2]
        else:
            border1 = int((1 - self.few_shot_ratio) * num_train) - self.seq_len
            border2 = num_train

        train_values = values[border1s[0] : border2s[0]]
        if self.scale:
            self.value_loc = np.nanmean(train_values, axis=0, keepdims=True)
            self.value_scale = np.nanstd(train_values, axis=0, keepdims=True)
            self.value_scale = np.where(self.value_scale == 0, 1.0, self.value_scale)
            values = (values - self.value_loc) / self.value_scale
            target_idx = value_columns.index(self.target)
            self.prior_loc = float(self.value_loc[0, target_idx])
            self.prior_scale = float(self.value_scale[0, target_idx])
            prior = (prior - self.prior_loc) / self.prior_scale
        else:
            self.value_loc = np.zeros((1, len(value_columns)), dtype=np.float32)
            self.value_scale = np.ones((1, len(value_columns)), dtype=np.float32)
            self.prior_loc = 0.0
            self.prior_scale = 1.0

        self.values = values[border1:border2]
        self.prior = prior[border1:border2]
        self.text = text[border1:border2]
        self.image_paths = image_paths[border1:border2] if image_paths is not None else None
        self.border1 = border1
        self.border2 = border2
        self.n_targets = len(self.target_columns)
        self.n_variates = len(value_columns)
        self.length = max(0, len(self.values) - self.seq_len - self.pred_len + 1)

    def __len__(self) -> int:
        return self.length

    def _load_image(self, raw_path: str) -> torch.Tensor:
        raw_path = raw_path.strip()
        if not raw_path:
            raise ValueError("TimeMMD image path is empty; image input must be provided explicitly")

        image_path = Path(raw_path)
        if not image_path.is_absolute():
            image_path = self.image_root_path / image_path
        if not image_path.exists():
            raise FileNotFoundError(f"TimeMMD image file not found: {image_path}")

        from PIL import Image

        with Image.open(image_path) as image:
            image = image.convert("L").resize((self.image_size, self.image_size), Image.Resampling.BILINEAR)
            array = np.asarray(image, dtype=np.float32) / 255.0
        return torch.from_numpy(array).unsqueeze(0)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor | int]:
        if index < 0 or index >= len(self):
            raise IndexError(index)

        context = self.values[index : index + self.seq_len].T
        future = self.values[index + self.seq_len : index + self.seq_len + self.pred_len].T
        future_target = future.copy()
        future_covariates = np.full_like(future, fill_value=np.nan)
        if self.n_targets < self.n_variates:
            future_target[self.n_targets :] = np.nan

        tokenized = self.tokenizer(
            self.text[index],
            padding="max_length",
            truncation=True,
            max_length=self.max_text_length,
            return_tensors="pt",
        )
        risk = self.prior[index + self.seq_len : index + self.seq_len + self.pred_len].reshape(self.pred_len, 1)
        item = {
            "context": torch.tensor(context, dtype=torch.float32),
            "future_target": torch.tensor(future_target, dtype=torch.float32),
            "future_covariates": torch.tensor(future_covariates, dtype=torch.float32),
            "risk_features": torch.tensor(risk, dtype=torch.float32),
            "text_input_ids": tokenized["input_ids"].squeeze(0).to(torch.long),
            "text_attention_mask": tokenized["attention_mask"].squeeze(0).to(torch.long),
            "text_token_type_ids": tokenized.get("token_type_ids", torch.zeros_like(tokenized["input_ids"]))
            .squeeze(0)
            .to(torch.long),
            "n_targets": self.n_targets,
        }
        if self.image_paths is not None:
            item["vision_values"] = self._load_image(self.image_paths[index])
        return item

    def inverse_transform(self, values: np.ndarray, column_indices: list[int] | None = None) -> np.ndarray:
        loc = self.value_loc
        scale = self.value_scale
        if column_indices is not None:
            loc = loc[:, column_indices]
            scale = scale[:, column_indices]
        return values * scale + loc


def build_timemmd_batch(items: list[dict[str, torch.Tensor | int]], output_patch_size: int) -> dict[str, Any]:
    batch_context = []
    batch_future_target = []
    batch_future_covariates = []
    batch_group_ids = []
    batch_text_input_ids = []
    batch_text_attention_mask = []
    batch_text_token_type_ids = []
    batch_risk_features = []
    batch_vision_values = []
    target_idx_ranges: list[tuple[int, int]] = []

    target_start_idx = 0
    for group_id, item in enumerate(items):
        context = item["context"]
        future_target = item["future_target"]
        future_covariates = item["future_covariates"]
        risk_features = item["risk_features"]
        assert isinstance(context, torch.Tensor)
        assert isinstance(future_target, torch.Tensor)
        assert isinstance(future_covariates, torch.Tensor)
        assert isinstance(risk_features, torch.Tensor)
        n_variates = context.shape[0]
        n_targets = int(item["n_targets"])

        batch_context.append(context)
        batch_future_target.append(future_target)
        batch_future_covariates.append(future_covariates)
        batch_group_ids.append(torch.full((n_variates,), fill_value=group_id, dtype=torch.long))
        target_idx_ranges.append((target_start_idx, target_start_idx + n_targets))
        target_start_idx += n_variates

        for key, target_list in [
            ("text_input_ids", batch_text_input_ids),
            ("text_attention_mask", batch_text_attention_mask),
            ("text_token_type_ids", batch_text_token_type_ids),
        ]:
            tensor = item[key]
            assert isinstance(tensor, torch.Tensor)
            target_list.append(tensor.unsqueeze(0).repeat(n_variates, 1))
        batch_risk_features.append(risk_features.unsqueeze(0).repeat(n_variates, 1, 1))
        vision_values = item.get("vision_values")
        if vision_values is not None:
            assert isinstance(vision_values, torch.Tensor)
            batch_vision_values.append(vision_values.unsqueeze(0).repeat(n_variates, 1, 1, 1))

    prediction_length = batch_future_target[0].shape[-1]
    batch = {
        "context": torch.cat(batch_context, dim=0),
        "future_target": torch.cat(batch_future_target, dim=0),
        "future_covariates": torch.cat(batch_future_covariates, dim=0),
        "group_ids": torch.cat(batch_group_ids, dim=0),
        "num_output_patches": math.ceil(prediction_length / output_patch_size),
        "target_idx_ranges": target_idx_ranges,
        "text_input_ids": torch.cat(batch_text_input_ids, dim=0),
        "text_attention_mask": torch.cat(batch_text_attention_mask, dim=0),
        "text_token_type_ids": torch.cat(batch_text_token_type_ids, dim=0),
        "risk_features": torch.cat(batch_risk_features, dim=0),
    }
    if batch_vision_values:
        batch["vision_values"] = torch.cat(batch_vision_values, dim=0)
    return batch


class TimeMMDBatchDataset(IterableDataset):
    def __init__(
        self,
        window_dataset: TimeMMDWindowDataset,
        *,
        batch_size: int,
        output_patch_size: int,
        shuffle: bool,
        repeat: bool,
    ) -> None:
        super().__init__()
        self.window_dataset = window_dataset
        self.batch_size = batch_size
        self.output_patch_size = output_patch_size
        self.shuffle = shuffle
        self.repeat = repeat
        if len(window_dataset) == 0:
            raise ValueError(
                "TimeMMD split contains no sliding windows; reduce seq_len/pred_len or use a longer CSV file"
            )

    def __iter__(self) -> Iterator[dict[str, Any]]:
        indices = np.arange(len(self.window_dataset))
        while True:
            if self.shuffle:
                np.random.shuffle(indices)
            for start in range(0, len(indices), self.batch_size):
                batch_indices = indices[start : start + self.batch_size]
                if len(batch_indices) == 0:
                    continue
                yield build_timemmd_batch(
                    [self.window_dataset[int(idx)] for idx in batch_indices],
                    self.output_patch_size,
                )
            if not self.repeat:
                break

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

REQUIRED_COLUMNS = {"date", "OT", "prior_history_avg", "start_date", "end_date", "fact"}
AURORA_MISSING_TEXT = "No information available"


def _test_window_count(n_rows: int, seq_len: int, pred_len: int) -> int:
    num_test = int(n_rows * 0.2)
    test_start = max(0, n_rows - num_test - seq_len)
    return max(0, n_rows - test_start - seq_len - pred_len + 1)


def validate_data_root(data_root: str | Path, tasks: list[dict[str, Any]]) -> dict[str, Any]:
    data_root = Path(data_root)
    results = []
    for task in tasks:
        path = data_root / task["data_path"]
        if not path.exists():
            raise FileNotFoundError(f"TimeMMD file not found: {path}")

        df = pd.read_csv(path)
        missing = REQUIRED_COLUMNS - set(df.columns)
        if missing:
            raise ValueError(f"{path.name} missing required columns: {sorted(missing)}")

        for col in ["date", "start_date", "end_date"]:
            parsed = pd.to_datetime(df[col], errors="coerce")
            if parsed.isna().any():
                raise ValueError(f"{path.name} has unparsable {col} values")

        target = pd.to_numeric(df["OT"], errors="coerce")
        if target.isna().any():
            raise ValueError(f"{path.name} has non-numeric OT values")
        if not np.isfinite(target.to_numpy(dtype=float)).all():
            raise ValueError(f"{path.name} has non-finite values in OT")

        fact = df["fact"].fillna(AURORA_MISSING_TEXT).astype(str)
        if fact.str.strip().eq("").any():
            raise ValueError(f"{path.name} contains empty fact text")

        n_windows = _test_window_count(len(df), int(task["seq_len"]), int(task["pred_len"]))
        if n_windows == 0:
            raise ValueError(
                f"{path.name} has no test windows for seq_len={task['seq_len']} pred_len={task['pred_len']}"
            )

        results.append(
            {
                "domain": task["domain"],
                "pred_len": int(task["pred_len"]),
                "data_path": task["data_path"],
                "n_rows": int(len(df)),
                "n_windows": int(n_windows),
            }
        )

    return {"ok": True, "data_root": str(data_root.resolve()), "tasks": results}

In [ ]:
from __future__ import annotations

import argparse
import csv
import io
import hashlib
import json
import math
import platform
import random
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from chronos_echo import Chronos2EchoConfig, Chronos2EchoPipeline


DEFAULT_CHECKPOINT_ROOT = Path("timemmd_checkpoints") / "chronos2_echo_fewshot"
SOURCE_URL = AURORA_SOURCE_URL

PROJECT2_ECHO_CONFIG: dict[str, Any] = {
    "vision_model_name_or_path": "google/vit-base-patch16-224",
    "vision_image_size": 224,
    "freeze_vision_backbone": True,
    "reconstruction_loss_weight": 0.5,
    "residual_scale_init": 1.0,
    "use_pseudo_image": False,
    "guard_against_baseline": False,
}


class NoopTokenizer:
    def __call__(self, text, *, padding, truncation, max_length, return_tensors):
        del text, padding, truncation
        if return_tensors != "pt":
            raise ValueError("NoopTokenizer only supports return_tensors='pt'")
        input_ids = torch.zeros((1, max_length), dtype=torch.long)
        return {
            "input_ids": input_ids,
            "attention_mask": torch.zeros_like(input_ids),
            "token_type_ids": torch.zeros_like(input_ids),
        }


def load_manifest(path: str | Path | None = None) -> list[dict[str, Any]]:
    if path is None:
        rows = list(csv.DictReader(io.StringIO(MANIFEST_CSV.strip())))
    else:
        with Path(path).open(newline="", encoding="utf-8") as fp:
            rows = list(csv.DictReader(fp))
    tasks: list[dict[str, Any]] = []
    for row in rows:
        task = dict(row)
        for key in ["seq_len", "pred_len", "inference_token_len", "batch_size"]:
            task[key] = int(task[key])
        tasks.append(task)
    return tasks


def set_reproducible(seed: int) -> dict[str, Any]:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    return {"seed": seed, "torch_deterministic_algorithms": True}


def aurora_metric_row(model: str, task: dict[str, Any], pred: np.ndarray, true: np.ndarray) -> dict[str, Any]:
    pred = np.asarray(pred, dtype=np.float64)
    true = np.asarray(true, dtype=np.float64)
    diff = pred - true
    mse = float(np.mean(diff**2))
    mae = float(np.mean(np.abs(diff)))
    with np.errstate(divide="ignore", invalid="ignore"):
        mape = float(np.mean(np.abs(diff / true)))
        mspe = float(np.mean(np.square(diff / true)))
    rse_denom = np.sqrt(np.sum((true - true.mean()) ** 2))
    rse = float(np.sqrt(np.sum(diff**2)) / rse_denom) if rse_denom else float("nan")
    u = ((true - true.mean(0)) * (pred - pred.mean(0))).sum(0)
    d = np.sqrt(((true - true.mean(0)) ** 2 * (pred - pred.mean(0)) ** 2).sum(0)) + 1e-12
    corr = float(np.asarray(0.01 * (u / d).mean(-1)).mean())
    return {
        "model": model,
        "domain": task["domain"],
        "pred_len": int(task["pred_len"]),
        "seq_len": int(task["seq_len"]),
        "mse": mse,
        "mae": mae,
        "rmse": float(math.sqrt(mse)),
        "mape": mape,
        "mspe": mspe,
        "rse": rse,
        "corr": corr,
        "n_windows": int(true.shape[0]) if true.ndim else int(true.size),
    }


def _as_numpy(value: Any) -> np.ndarray:
    if isinstance(value, list):
        if value and torch.is_tensor(value[0]):
            value = torch.stack([item.detach().cpu() for item in value])
        else:
            value = np.asarray(value)
    if torch.is_tensor(value):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def _median_from_quantiles(quantiles: Any, means: Any) -> np.ndarray:
    means_array = _as_numpy(means)
    if means_array.ndim == 3 and means_array.shape[1] == 1:
        means_array = means_array[:, 0, :]
    if means_array.ndim == 2:
        return means_array

    quantile_array = _as_numpy(quantiles)
    if quantile_array.ndim == 4 and quantile_array.shape[1] == 1:
        quantile_array = quantile_array[:, 0, :, :]
    if quantile_array.ndim == 3:
        return quantile_array[..., quantile_array.shape[-1] // 2]
    return quantile_array


def _timemmd_window_dataset(task: dict[str, Any], data_root: Path, tokenizer: Any) -> TimeMMDWindowDataset:
    return TimeMMDWindowDataset(
        root_path=data_root,
        data_path=task["data_path"],
        flag=task["split"],
        seq_len=task["seq_len"],
        pred_len=task["pred_len"],
        target=task["target"],
        features=task["features"],
        tokenizer=tokenizer,
        text_column=task["text_column"],
    )


def _echo_tokenizer(pipeline: Chronos2EchoPipeline, tokenizer: Any | None = None) -> Any:
    if tokenizer is not None:
        return tokenizer
    config = pipeline._unwrap_echo_model(pipeline.model).echo_config
    return create_timemmd_tokenizer(config.text_tokenizer_name_or_path or config.text_model_name_or_path)


def _echo_timemmd_dataset(
    pipeline: Chronos2EchoPipeline,
    task: dict[str, Any],
    *,
    data_root: Path,
    flag: str,
    batch_size: int,
    shuffle: bool,
    repeat: bool,
    tokenizer: Any | None = None,
    few_shot_ratio: float = 0.1,
) -> tuple[TimeMMDWindowDataset, TimeMMDBatchDataset]:
    echo_config = pipeline._unwrap_echo_model(pipeline.model).echo_config
    window_dataset = TimeMMDWindowDataset(
        root_path=data_root,
        data_path=task["data_path"],
        flag=flag,
        seq_len=task["seq_len"],
        pred_len=task["pred_len"],
        target=task["target"],
        features=task["features"],
        tokenizer=_echo_tokenizer(pipeline, tokenizer),
        max_text_length=echo_config.max_text_length,
        text_column=task.get("text_column", "fact"),
        image_size=echo_config.vision_image_size,
        few_shot_ratio=few_shot_ratio,
    )
    return window_dataset, TimeMMDBatchDataset(
        window_dataset,
        batch_size=batch_size,
        output_patch_size=pipeline.model_output_patch_size,
        shuffle=shuffle,
        repeat=repeat,
    )


def validate_fewshot_splits(data_root: Path, tasks: list[dict[str, Any]]) -> None:
    bad = []
    for task in tasks:
        counts = {}
        for flag in ["fewshot", "val"]:
            counts[flag] = len(
                TimeMMDWindowDataset(
                    root_path=data_root,
                    data_path=task["data_path"],
                    flag=flag,
                    seq_len=task["seq_len"],
                    pred_len=task["pred_len"],
                    target=task["target"],
                    features=task["features"],
                    tokenizer=NoopTokenizer(),
                    text_column=task["text_column"],
                )
            )
        if counts["fewshot"] == 0 or counts["val"] == 0:
            bad.append(f"{task['domain']} pred_len={task['pred_len']} fewshot={counts['fewshot']} val={counts['val']}")
    if bad:
        raise ValueError("Aurora 10% few-shot split has no training/eval windows for: " + "; ".join(bad))


def evaluate_chronos2(
    model_name: str,
    tasks: list[dict[str, Any]],
    *,
    data_root: Path,
    model_path: str,
    device: str,
    batch_size: int | None = None,
    **_: Any,
) -> list[dict[str, Any]]:
    from chronos import Chronos2Pipeline

    pipeline = Chronos2Pipeline.from_pretrained(model_path, device_map=device)
    rows = []
    for task in tasks:
        dataset = _timemmd_window_dataset(task, data_root, NoopTokenizer())
        preds = []
        trues = []
        bs = batch_size or task["batch_size"]
        for start in range(0, len(dataset), bs):
            items = [dataset[idx] for idx in range(start, min(start + bs, len(dataset)))]
            context = torch.stack([item["context"] for item in items])
            quantiles, means = pipeline.predict_quantiles(
                context,
                prediction_length=task["pred_len"],
                quantile_levels=[0.5],
                limit_prediction_length=False,
            )
            pred = _median_from_quantiles(quantiles, means)[..., : task["pred_len"]]
            true = torch.stack([item["future_target"].squeeze(0) for item in items]).numpy()
            preds.append(np.asarray(pred).reshape(len(items), task["pred_len"], 1))
            trues.append(true.reshape(len(items), task["pred_len"], 1))
        rows.append(aurora_metric_row(model_name, task, np.concatenate(preds), np.concatenate(trues)))
    return rows


def _default_tokenizer_path() -> str:
    if not TEXT_TOKENIZER:
        raise ValueError("Set TEXT_TOKENIZER explicitly for ECHO text input.")
    return str(TEXT_TOKENIZER)


def resolve_echo_config(args: argparse.Namespace) -> dict[str, Any]:
    config = dict(PROJECT2_ECHO_CONFIG)
    if args.echo_config:
        config.update(json.loads(args.echo_config))
    config.setdefault("text_tokenizer_name_or_path", args.text_tokenizer or _default_tokenizer_path())
    return config


def load_echo_pipeline(model_path: str | Path, echo_config: dict[str, Any], device: str) -> Chronos2EchoPipeline:
    model_path = Path(model_path) if not str(model_path).startswith("amazon/") else model_path
    is_peft = (
        isinstance(model_path, Path)
        and model_path.is_dir()
        and (model_path / "adapter_config.json").is_file()
        and not (model_path / "model.safetensors").is_file()
    )
    if not is_peft:
        return Chronos2EchoPipeline.from_pretrained(model_path, echo_config=Chronos2EchoConfig(**echo_config), device_map=device)

    from peft import PeftModel

    adapter_config = json.loads((model_path / "adapter_config.json").read_text(encoding="utf-8"))
    base_model = adapter_config.get("base_model_name_or_path", "amazon/chronos-2")
    base_config = Chronos2EchoConfig(**echo_config)
    base_config.residual_scale_init = 0.0
    base_config.guard_against_baseline = False
    base_pipeline = Chronos2EchoPipeline.from_pretrained(base_model, echo_config=base_config, device_map=device)
    model = PeftModel.from_pretrained(base_pipeline.model, str(model_path))
    return Chronos2EchoPipeline(model=model)


def _evaluate_echo_pipeline(
    model_name: str,
    tasks: list[dict[str, Any]],
    *,
    data_root: Path,
    pipeline_for_task,
    batch_size: int | None = None,
) -> list[dict[str, Any]]:
    rows = []
    for task in tasks:
        pipeline = pipeline_for_task(task)
        _, dataset = _echo_timemmd_dataset(
            pipeline,
            task,
            data_root=data_root,
            flag=task["split"],
            batch_size=batch_size or task["batch_size"],
            shuffle=False,
            repeat=False,
        )
        device = pipeline._unwrap_echo_model(pipeline.model).device
        loader = DataLoader(dataset, batch_size=None, pin_memory=device.type == "cuda")
        quantile_windows = []
        target_windows = []
        median_idx = min(range(len(pipeline.quantiles)), key=lambda idx: abs(float(pipeline.quantiles[idx]) - 0.5))
        pipeline.model.eval()
        with torch.no_grad():
            for batch in loader:
                target_idx_ranges = batch.pop("target_idx_ranges")
                future_target = batch["future_target"]
                model_inputs = {}
                for key, value in batch.items():
                    if key in {"future_target", "risk_features"}:
                        continue
                    model_inputs[key] = value.to(device) if torch.is_tensor(value) else value
                output = pipeline.model(**model_inputs)
                quantile_preds = output.quantile_preds[..., : task["pred_len"]].cpu()
                for start, end in target_idx_ranges:
                    quantile_windows.append(quantile_preds[start:end].permute(2, 0, 1))
                    target_windows.append(future_target[start:end].permute(1, 0))
        quantiles = torch.stack(quantile_windows, dim=0).numpy()
        targets = torch.stack(target_windows, dim=0).numpy()
        rows.append(aurora_metric_row(model_name, task, quantiles[..., median_idx], targets))
    return rows


def evaluate_echo_zero_shot(
    model_name: str,
    tasks: list[dict[str, Any]],
    *,
    data_root: Path,
    model_path: str,
    device: str,
    echo_config: dict[str, Any],
    batch_size: int | None = None,
    **_: Any,
) -> list[dict[str, Any]]:
    pipeline = load_echo_pipeline(model_path, echo_config, device)
    return _evaluate_echo_pipeline(
        model_name,
        tasks,
        data_root=data_root,
        pipeline_for_task=lambda task: pipeline,
        batch_size=batch_size,
    )


def _file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fp:
        for chunk in iter(lambda: fp.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _task_checkpoint_dir(checkpoint_root: Path, task: dict[str, Any]) -> Path:
    return checkpoint_root / task["domain"] / f"H{task['seq_len']}_F{task['pred_len']}" / "finetuned-ckpt"


def _task_fingerprint(task: dict[str, Any], data_root: Path, echo_config: dict[str, Any], args: argparse.Namespace) -> dict[str, Any]:
    return {
        "task": {key: task[key] for key in ["domain", "data_path", "seq_len", "pred_len", "features", "target"]},
        "data_sha256": _file_sha256(data_root / task["data_path"]),
        "echo_config": echo_config,
        "training": {
            "learning_rate": args.fewshot_lr,
            "num_steps": args.fewshot_steps,
            "warmup_steps": args.fewshot_warmup_steps,
            "batch_size": args.fewshot_batch_size,
            "lr_scheduler_type": args.fewshot_lr_scheduler,
            "finetune_mode": "lora",
            "flag": "fewshot",
            "few_shot_ratio": 0.1,
        },
    }


def fit_echo_timemmd(
    pipeline: Chronos2EchoPipeline,
    *,
    data_root: Path,
    task: dict[str, Any],
    batch_size: int,
    learning_rate: float,
    num_steps: int,
    warmup_steps: int,
    lr_scheduler_type: str,
    output_dir: Path,
    finetuned_ckpt_name: str = "finetuned-ckpt",
) -> None:
    from transformers.training_args import TrainingArguments

    from chronos_echo.trainer import Chronos2Trainer, EvaluateAndSaveFinalStepCallback

    train_pipeline = pipeline.fit_echo(finetune_mode="lora")
    train_base_model = train_pipeline._unwrap_echo_model(train_pipeline.model)
    _, train_dataset = _echo_timemmd_dataset(
        train_pipeline,
        task,
        data_root=data_root,
        flag="fewshot",
        batch_size=batch_size,
        shuffle=True,
        repeat=True,
        few_shot_ratio=0.1,
    )
    _, eval_dataset = _echo_timemmd_dataset(
        train_pipeline,
        task,
        data_root=data_root,
        flag="val",
        batch_size=batch_size,
        shuffle=False,
        repeat=False,
        few_shot_ratio=0.1,
    )

    use_cpu = str(train_base_model.device) == "cpu"
    has_sm80 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    training_args = TrainingArguments(
        output_dir=str(output_dir),
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        lr_scheduler_type=lr_scheduler_type,
        warmup_steps=warmup_steps,
        optim="adamw_torch",
        logging_strategy="steps",
        logging_steps=100,
        disable_tqdm=False,
        report_to="none",
        max_steps=num_steps,
        dataloader_num_workers=0,
        tf32=has_sm80 and not use_cpu,
        bf16=False,
        save_only_model=True,
        prediction_loss_only=True,
        save_total_limit=1,
        save_strategy="steps",
        save_steps=100,
        eval_strategy="steps",
        eval_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        label_names=["future_target"],
        use_cpu=use_cpu,
        remove_unused_columns=False,
    )
    if not use_cpu:
        training_args._n_gpu = 1

    trainer = Chronos2Trainer(
        model=train_pipeline.model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        callbacks=[EvaluateAndSaveFinalStepCallback()],
    )
    trainer.train()

    trained_model = trainer.model
    trained_base_model = train_pipeline._unwrap_echo_model(trained_model)
    trained_base_model.chronos_config.context_length = max(trained_base_model.chronos_config.context_length, task["seq_len"])
    trained_base_model.chronos_config.max_output_patches = max(
        trained_base_model.chronos_config.max_output_patches,
        (task["pred_len"] + train_pipeline.model_output_patch_size - 1) // train_pipeline.model_output_patch_size,
    )
    if hasattr(trained_base_model, "config"):
        trained_base_model.config.chronos_config = trained_base_model.chronos_config.__dict__
        trained_base_model.config.echo_config = trained_base_model.echo_config.__dict__
        trained_base_model.config.chronos_pipeline_class = "Chronos2EchoPipeline"
        trained_base_model.config.architectures = ["Chronos2EchoModel"]

    Chronos2EchoPipeline(model=trained_model).save_pretrained(output_dir / finetuned_ckpt_name)  # type: ignore[arg-type]


def train_echo_fewshot(task: dict[str, Any], *, data_root: Path, checkpoint_root: Path, args: argparse.Namespace, echo_config: dict[str, Any]) -> None:
    checkpoint_dir = _task_checkpoint_dir(checkpoint_root, task)
    output_dir = checkpoint_dir.parent
    fingerprint = _task_fingerprint(task, data_root, echo_config, args)
    fingerprint_path = output_dir / "fingerprint.json"
    if (
        not args.force_retrain
        and checkpoint_dir.exists()
        and fingerprint_path.exists()
        and json.loads(fingerprint_path.read_text(encoding="utf-8")) == fingerprint
    ):
        return

    output_dir.mkdir(parents=True, exist_ok=True)
    pipeline = Chronos2EchoPipeline.from_pretrained(
        args.echo_base_model,
        echo_config=Chronos2EchoConfig(**echo_config),
        device_map=args.device,
    )
    fit_echo_timemmd(
        pipeline,
        data_root=data_root,
        task=task,
        batch_size=args.fewshot_batch_size,
        learning_rate=args.fewshot_lr,
        num_steps=args.fewshot_steps,
        warmup_steps=args.fewshot_warmup_steps,
        lr_scheduler_type=args.fewshot_lr_scheduler,
        output_dir=output_dir,
        finetuned_ckpt_name="finetuned-ckpt",
    )
    fingerprint_path.write_text(json.dumps(fingerprint, indent=2), encoding="utf-8")


def evaluate_echo_fewshot(
    model_name: str,
    tasks: list[dict[str, Any]],
    *,
    data_root: Path,
    checkpoint_root: Path,
    device: str,
    echo_config: dict[str, Any],
    batch_size: int | None = None,
    **_: Any,
) -> list[dict[str, Any]]:
    cache: dict[Path, Chronos2EchoPipeline] = {}

    def pipeline_for_task(task: dict[str, Any]) -> Chronos2EchoPipeline:
        checkpoint_dir = _task_checkpoint_dir(checkpoint_root, task)
        if checkpoint_dir not in cache:
            cache[checkpoint_dir] = load_echo_pipeline(checkpoint_dir, echo_config, device)
        return cache[checkpoint_dir]

    return _evaluate_echo_pipeline(
        model_name,
        tasks,
        data_root=data_root,
        pipeline_for_task=pipeline_for_task,
        batch_size=batch_size,
    )


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    if rows:
        pd.DataFrame(rows).to_csv(path, index=False)
    else:
        path.write_text("", encoding="utf-8")


def comparison_rows(metrics: list[dict[str, Any]]) -> list[dict[str, Any]]:
    rows = []
    for row in metrics:
        reference_model = "aurora_few_shot" if row["model"] == "chronos2_echo_few_shot" else "aurora_zero_shot"
        reference = get_reference(row["domain"], int(row["pred_len"]), reference_model)
        if reference is None:
            continue
        rows.append(
            {
                "model": row["model"],
                "domain": row["domain"],
                "pred_len": row["pred_len"],
                "reference_model": reference_model,
                "model_mse": row["mse"],
                "reference_mse": reference["mse"],
                "delta_mse": row["mse"] - reference["mse"],
                "relative_mse_change": (row["mse"] - reference["mse"]) / reference["mse"] if reference["mse"] else float("nan"),
                "model_mae": row["mae"],
                "reference_mae": reference["mae"],
                "delta_mae": row["mae"] - reference["mae"],
                "relative_mae_change": (row["mae"] - reference["mae"]) / reference["mae"] if reference["mae"] else float("nan"),
            }
        )
    return rows


def domain_summary_rows(metrics: list[dict[str, Any]]) -> list[dict[str, Any]]:
    comparisons = pd.DataFrame(comparison_rows(metrics))
    if comparisons.empty:
        return []
    grouped = comparisons.groupby(["model", "domain", "reference_model"], as_index=False).agg(
        mean_model_mse=("model_mse", "mean"),
        mean_reference_mse=("reference_mse", "mean"),
        mean_delta_mse=("delta_mse", "mean"),
        mean_model_mae=("model_mae", "mean"),
        mean_reference_mae=("reference_mae", "mean"),
        mean_delta_mae=("delta_mae", "mean"),
    )
    return grouped.to_dict(orient="records")

In [ ]:
def write_repro(path: Path, args: argparse.Namespace, repro: dict[str, Any]) -> None:
    lines = [
        "runner: timemmd_colab_benchmark.ipynb",
        f"python: {platform.python_version()}",
        f"torch: {torch.__version__}",
        f"cuda_available: {torch.cuda.is_available()}",
        f"device: {args.device}",
        f"data_root: {args.data_root}",
        f"output_dir: {args.output_dir}",
        f"models: {args.models}",
        f"aurora_reference: {SOURCE_URL}",
        f"seed: {repro['seed']}",
        f"torch_deterministic_algorithms: {repro['torch_deterministic_algorithms']}",
        f"package_spec: {PACKAGE_SPEC}",
    ]
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def run_timemmd_benchmark() -> Path:
    args = argparse.Namespace(
        data_root=str(DATA_ROOT),
        manifest=None,
        output_dir=str(OUTPUT_DIR),
        models=MODELS,
        chronos_model=CHRONOS_MODEL,
        echo_base_model=ECHO_BASE_MODEL,
        echo_config=None,
        text_tokenizer=TEXT_TOKENIZER,
        device=DEVICE,
        seed=SEED,
        batch_size=BATCH_SIZE,
        checkpoint_root=str(CHECKPOINT_ROOT),
        fewshot_lr=FEWSHOT_LR,
        fewshot_steps=FEWSHOT_STEPS,
        fewshot_warmup_steps=FEWSHOT_WARMUP_STEPS,
        fewshot_batch_size=FEWSHOT_BATCH_SIZE,
        fewshot_lr_scheduler=FEWSHOT_LR_SCHEDULER,
        force_retrain=FORCE_RETRAIN,
        dry_run=False,
    )
    tasks = load_manifest()
    validation = validate_data_root(Path(args.data_root), tasks)
    selected = {item.strip() for item in args.models.split(",") if item.strip()}
    unknown = selected - {"chronos2", "echo_zero_shot", "echo_few_shot"}
    if unknown:
        raise ValueError(f"Unsupported MODELS entries: {sorted(unknown)}")
    if not selected:
        raise ValueError("MODELS must select at least one model.")

    echo_config = dict(PROJECT2_ECHO_CONFIG)
    echo_config.update(ECHO_CONFIG_OVERRIDES)
    args.echo_config = json.dumps(echo_config)
    echo_config = resolve_echo_config(args)
    if "echo_few_shot" in selected:
        validate_fewshot_splits(Path(args.data_root), tasks)

    repro = set_reproducible(args.seed)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "input_validation.json").write_text(json.dumps(validation, indent=2), encoding="utf-8")

    metrics: list[dict[str, Any]] = []
    if "chronos2" in selected:
        metrics.extend(
            evaluate_chronos2(
                "chronos2",
                tasks,
                data_root=Path(args.data_root),
                model_path=args.chronos_model,
                device=args.device,
                batch_size=args.batch_size,
            )
        )
    if "echo_zero_shot" in selected:
        metrics.extend(
            evaluate_echo_zero_shot(
                "chronos2_echo_zero_shot",
                tasks,
                data_root=Path(args.data_root),
                model_path=args.echo_base_model,
                device=args.device,
                echo_config=echo_config,
                batch_size=args.batch_size,
            )
        )
    if "echo_few_shot" in selected:
        checkpoint_root = Path(args.checkpoint_root)
        for task in tasks:
            train_echo_fewshot(task, data_root=Path(args.data_root), checkpoint_root=checkpoint_root, args=args, echo_config=echo_config)
        metrics.extend(
            evaluate_echo_fewshot(
                "chronos2_echo_few_shot",
                tasks,
                data_root=Path(args.data_root),
                checkpoint_root=checkpoint_root,
                device=args.device,
                echo_config=echo_config,
                batch_size=args.batch_size,
            )
        )

    comparisons = comparison_rows(metrics)
    summaries = domain_summary_rows(metrics)
    write_csv(output_dir / "metrics.csv", metrics)
    write_csv(output_dir / "comparison.csv", comparisons)
    write_csv(output_dir / "domain_summary.csv", summaries)
    summary = {
        "config": vars(args),
        "aurora_reference": SOURCE_URL,
        "reproducibility": repro,
        "n_tasks": len(tasks),
        "n_metrics": len(metrics),
        "n_comparisons": len(comparisons),
        "models": sorted(selected),
        "echo_config": echo_config,
    }
    (output_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    write_repro(output_dir / "repro.txt", args, repro)
    return output_dir

## Run

In [ ]:
RUN_OUTPUT_DIR = run_timemmd_benchmark()
print("Wrote benchmark outputs to", RUN_OUTPUT_DIR)

## Results

In [ ]:
from IPython.display import display

for name in ["metrics.csv", "comparison.csv", "domain_summary.csv"]:
    path = RUN_OUTPUT_DIR / name
    print("\n" + name)
    if path.exists() and path.stat().st_size > 0:
        display(pd.read_csv(path).head(20))
    else:
        print("empty")

print("summary.json")
print((RUN_OUTPUT_DIR / "summary.json").read_text(encoding="utf-8")[:4000])

In [ ]:
archive_path = shutil.make_archive(str(RUN_OUTPUT_DIR), "zip", root_dir=RUN_OUTPUT_DIR)
print("Packaged", archive_path)

try:
    from google.colab import files
    files.download(archive_path)
except Exception as exc:
    print("Download skipped:", exc)
    print("Archive path:", archive_path)